In [1]:
import yfinance as yf
import pandas as pd
from fredapi import Fred
import os

In [2]:
# ── CONFIG ─────────────────────────────────────────────
START = "2015-01-01"
END   = "2026-05-23"
FRED_KEY = "7ba183bf78127d55d1df4d48ef259f10"   
# ───────────────────────────────────────────────────────

os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

In [3]:
# ── 1. MARKET DATA VIA YFINANCE ────────────────────────
tickers = {
    "USDINR":    "USDINR=X",   # USD/INR exchange rate
    "NIFTY":     "^NSEI",      # Nifty 50
    "INDIAVIX":  "^INDIAVIX",  # India Volatility Index
    "GOLD":      "GC=F",       # Gold Futures (USD)
    "CRUDE":     "CL=F",       # Crude Oil WTI (USD)
    "DXY":       "DX-Y.NYB",   # US Dollar Index
}

market_data = {}
for name, ticker in tickers.items():
    print(f"Downloading {name}...")
    df = yf.download(ticker, start=START, end=END, 
                     auto_adjust=True, progress=False)
    
    # Handle MultiIndex columns (yfinance >= 0.2.x)
    if isinstance(df.columns, pd.MultiIndex):
        close = df["Close"].iloc[:, 0]
    else:
        close = df["Close"]
    
    # Ensure it's a Series, not a single-column DataFrame
    if isinstance(close, pd.DataFrame):
        close = close.squeeze()
    
    market_data[name] = close.rename(name)
    print(f"  ✓ {name}: {len(close)} rows, latest = {close.dropna().iloc[-1]:.4f}")


# Merge all into one DataFrame
market_df = pd.concat(market_data.values(), axis=1)
market_df.index = pd.to_datetime(market_df.index)
market_df.index = market_df.index.tz_localize(None)  # remove timezone
market_df.dropna(how='all', inplace=True)
market_df.to_csv("../data/raw/market_data.csv")
print(f"Market data saved: {market_df.shape}")
print(market_df.tail(3))

  ✓ USDINR: 2964 rows, latest = 96.1739
  ✓ NIFTY: 2802 rows, latest = 23719.3008
  ✓ INDIAVIX: 2791 rows, latest = 17.9100
  ✓ GOLD: 2863 rows, latest = 4521.0000
  ✓ CRUDE: 2864 rows, latest = 96.6000
  ✓ DXY: 2865 rows, latest = 99.3200
Market data saved: (2971, 6)
               USDINR         NIFTY   INDIAVIX         GOLD      CRUDE  \
2026-05-20  96.565804  23659.000000  18.440001  4531.299805  98.260002   
2026-05-21  96.528297  23654.699219  17.820000  4539.799805  96.349998   
2026-05-22  96.173897  23719.300781  17.910000  4521.000000  96.599998   

                  DXY  
2026-05-20  99.110001  
2026-05-21  99.190002  
2026-05-22  99.320000  


C:\Users\srrml\AppData\Local\Temp\ipykernel_35652\439878139.py:32: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  market_df = pd.concat(market_data.values(), axis=1)


In [4]:
# ── 2. MACRO DATA VIA FRED ─────────────────────────────
fred = Fred(api_key=FRED_KEY)

fred_series = {
    "US_CPI":       "CPIAUCSL",    # US Consumer Price Index
    "US_FEDFUNDS":  "FEDFUNDS",    # US Federal Funds Rate
    "US_10Y":       "GS10",        # US 10-Year Treasury Yield
}

macro_data = {}
for name, series_id in fred_series.items():
    print(f"Downloading {name} from FRED...")
    s = fred.get_series(series_id, observation_start=START, observation_end=END)
    macro_data[name] = s.rename(name)

macro_df = pd.DataFrame(macro_data)
macro_df.index = pd.to_datetime(macro_df.index)
macro_df.to_csv("../data/raw/macro_fred.csv")
print(f"FRED macro data saved: {macro_df.shape}")

FRED macro data saved: (136, 3)


In [5]:
# ── 3. RBI REPO RATE — hardcoded from RBI announcements ─
# Source: RBI Monetary Policy announcements (public record)
rbi_repo = pd.DataFrame({
    "date": [
        "2015-01-01","2015-03-04","2015-06-02","2015-09-29",
        "2016-04-05","2016-10-04",
        "2018-06-06","2018-08-01",
        "2019-02-07","2019-04-04","2019-06-06","2019-08-07","2019-10-04",
        "2020-03-27","2020-05-22",
        "2022-05-04","2022-06-08","2022-08-05","2022-09-30","2022-12-07",
        "2023-02-08",
        "2024-10-09",
        "2025-02-07","2025-04-09","2025-06-06"
    ],
    "RBI_REPO": [
        7.75, 7.50, 7.25, 6.75,
        6.50, 6.25,
        6.25, 6.50,
        6.25, 6.00, 5.75, 5.40, 5.15,
        4.40, 4.00,
        4.40, 4.90, 5.40, 5.90, 6.25,
        6.50,
        6.25,
        6.25, 6.00, 5.75
    ]
}).set_index("date")
rbi_repo.index = pd.to_datetime(rbi_repo.index)
rbi_repo = rbi_repo.resample("D").ffill()
rbi_repo.to_csv("../data/raw/rbi_repo.csv")
print("RBI Repo Rate saved.")

RBI Repo Rate saved.


In [6]:
# ── 4. GEOPOLITICAL EVENTS — your key dataset ──────────
# These are the events your event study will analyse
events = pd.DataFrame([
    # ── INDIA-SPECIFIC (Direct channel on INR) ──────────
    {"date":"2016-09-29","event":"Uri Surgical Strikes",
     "category":"India-Pak","channel":"Direct FX",
     "note":"Immediate risk premium on INR"},

    {"date":"2019-02-14","event":"Pulwama Attack",
     "category":"India-Pak","channel":"Direct FX",
     "note":"Escalation fear; FII pause"},

    {"date":"2019-02-26","event":"Balakot Airstrike",
     "category":"India-Pak","channel":"Direct FX",
     "note":"Military escalation; INR volatile"},

    {"date":"2020-06-15","event":"Galwan Valley Clash",
     "category":"India-China","channel":"Direct FX + Trade",
     "note":"Trade decoupling risk added to direct FX pressure"},

    {"date":"2025-05-07","event":"Operation Sindoor",
     "category":"India-Pak","channel":"Direct FX",
     "note":"Verify exact date and update"},

    # ── MIDDLE EAST / STRAIT OF HORMUZ (Oil Channel) ────
    {"date":"2019-05-12","event":"Gulf of Oman Tanker Attacks",
     "category":"Strait of Hormuz","channel":"Oil Supply",
     "note":"Iran-attributed; crude spiked; India import risk"},

    {"date":"2019-09-14","event":"Saudi Aramco Drone Attack",
     "category":"Middle East","channel":"Oil Supply",
     "note":"5% of global supply disrupted overnight; crude +15%"},

    {"date":"2020-01-03","event":"Soleimani Killing",
     "category":"US-Iran","channel":"Oil Supply + Risk-Off",
     "note":"Strait closure threat; crude spike; global risk-off"},

    {"date":"2023-10-07","event":"Israel-Hamas War",
     "category":"Middle East","channel":"Oil Supply + Risk-Off",
     "note":"Middle East escalation; oil risk premium; FII outflows"},

    {"date":"2023-11-19","event":"Houthi Red Sea Attacks",
     "category":"Strait of Hormuz","channel":"Trade Route + Oil",
     "note":"15% of global trade disrupted; shipping costs surged"},

    # ── RUSSIA-UKRAINE (Multi-Channel) ──────────────────
    {"date":"2022-02-24","event":"Russia-Ukraine War Begins",
     "category":"Europe","channel":"Oil + Gas + Risk-Off + DXY",
     "note":"Crude +40%; global risk-off; dollar surge; worst EM sell-off"},

    {"date":"2022-10-05","event":"OPEC+ Production Cuts",
     "category":"Global Energy","channel":"Oil Supply",
     "note":"2M barrel/day cut; crude re-spiked; India CAD pressure"},

    # ── GLOBAL RISK-OFF (Risk Sentiment Channel) ─────────
    {"date":"2015-08-24","event":"China Stock Market Crash",
     "category":"Global","channel":"Risk-Off + DXY",
     "note":"Global EM selloff; INR hit alongside other EMs"},

    {"date":"2020-03-23","event":"COVID Global Lockdown",
     "category":"Global","channel":"Risk-Off + Oil Collapse",
     "note":"Oil went negative; global risk-off; INR record low"},
])

events["date"] = pd.to_datetime(events["date"])
events.to_csv("../data/raw/geopolitical_events.csv", index=False)
print(f"Saved {len(events)} events across 4 channels")
print(events.groupby("channel")["event"].count())

Saved 14 events across 4 channels
channel
Direct FX                     4
Direct FX + Trade             1
Oil + Gas + Risk-Off + DXY    1
Oil Supply                    3
Oil Supply + Risk-Off         2
Risk-Off + DXY                1
Risk-Off + Oil Collapse       1
Trade Route + Oil             1
Name: event, dtype: int64
